In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Full MILP toolkit (Gurobi) for:
  (A) XOR-differentials (your original style)
  (B) Restricted rotational XOR-differentials: enforce Δ = rot(Δ,k)
  (C) Linear trails (Fu-style add model + correct fan-out constraints)

NOTE:
- This is a *bit-level* model. It grows fast.
- Start with nbits=8/16 and small rounds to sanity-check.
"""

import re
from typing import Dict, Tuple, Optional, Iterable, List
import gurobipy as gp


# ----------------- bit helpers -----------------
def rotl(bits, r: int):
    n = len(bits); r %= n
    return [bits[(i - r) % n] for i in range(n)]

def rotr(bits, r: int):
    n = len(bits); r %= n
    return [bits[(i + r) % n] for i in range(n)]

def bits_from_word(m: gp.Model, name: str, nbits: int):
    return [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}[{i}]") for i in range(nbits)]

def eq_word(m: gp.Model, a, b, name: str):
    """Enforce a == b bitwise."""
    assert len(a) == len(b)
    for i in range(len(a)):
        m.addConstr(a[i] == b[i], name=f"{name}[{i}]")


# ============================================================
#  XOR-differential primitives (your style)
# ============================================================
def add_word(m: gp.Model, x, y, name: str):
    """Mod-2^n add encoded exactly per bit: x + y + cin = z + 2*cout, cin=0."""
    n = len(x)
    z = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_z[{i}]") for i in range(n)]
    c = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_c[{i}]") for i in range(n+1)]
    m.addConstr(c[0] == 0, name=f"{name}_cin0")
    for i in range(n):
        m.addConstr(x[i] + y[i] + c[i] == z[i] + 2*c[i+1], name=f"{name}_fa[{i}]")
    return z, c[1:]  # carry-outs c[1]..c[n] (c[n] is overflow)

def xor2_word(m: gp.Model, a, b, name: str):
    """Bitwise XOR via linearization: a + b = z + 2*(a&b)."""
    n = len(a)
    z = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_z[{i}]") for i in range(n)]
    t = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_t[{i}]") for i in range(n)]
    for i in range(n):
        m.addConstr(t[i] <= a[i]); m.addConstr(t[i] <= b[i]); m.addConstr(t[i] >= a[i] + b[i] - 1)
        m.addConstr(a[i] + b[i] == z[i] + 2*t[i], name=f"{name}_xor[{i}]")
    return z


def build_fha_xordiff_milp(
    rounds: int = 6,
    nbits: int = 64,
    lanes: int = 8,
    ROT_A: Optional[Iterable[int]] = None,
    ROT_B: Optional[Iterable[int]] = None,
    ROT_G: Optional[Iterable[int]] = None,
    iterative: bool = True,
    pins: Optional[Dict[str, int]] = None,
    mode: str = "min",                   # 'min' | 'feas_ge' | 'feas_le'
    target_weight: int = 255,            # used by feas modes
    cube_t0_bits: int = 0,
    cube_t0_value: int = 0,
    threads: int = 1,
    timelimit: Optional[int] = None,
    env: Optional[gp.Env] = None,
) -> Tuple[gp.Model, gp.LinExpr]:
    """
    XOR-differential MILP. Weight = sum of carry-outs from Mix-1 adds (overflow excluded).
    """
    if ROT_A is None: ROT_A = [63,19,39,15,23,31,11,5]
    if ROT_B is None: ROT_B = [21, 3,31,33,45, 5,17,7]
    if ROT_G is None: ROT_G = [25,45,51, 5,63,47, 7,1]
    ROT_A = list(ROT_A); ROT_B = list(ROT_B); ROT_G = list(ROT_G)
    if len(ROT_A) < lanes: ROT_A = (ROT_A * ((lanes + len(ROT_A)-1)//len(ROT_A)))[:lanes]
    if len(ROT_B) < lanes: ROT_B = (ROT_B * ((lanes + len(ROT_B)-1)//len(ROT_B)))[:lanes]
    if len(ROT_G) < lanes: ROT_G = (ROT_G * ((lanes + len(ROT_G)-1)//len(ROT_G)))[:lanes]

    m = gp.Model("fha_xordiff", env=env) if env else gp.Model("fha_xordiff")
    if threads:   m.Params.Threads   = threads
    if timelimit: m.Params.TimeLimit = timelimit

    L = [[bits_from_word(m, f"L{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]
    R = [[bits_from_word(m, f"R{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]

    weight_bits: List[gp.Var] = []

    for i in range(rounds):
        # 1) L_{i+1} = R_i
        for j in range(lanes):
            for b in range(nbits):
                m.addConstr(L[j][i+1][b] == R[j][i][b], name=f"copy_L[{j},{i},{b}]")

        # 2) F(R_i)
        V1, V2, V3 = [[] for _ in range(lanes)], [[] for _ in range(lanes)], [[] for _ in range(lanes)]
        for j in range(lanes):
            j1, j2 = (j+1) % lanes, (j+2) % lanes
            z1, carries = add_word(m, R[j][i], rotl(R[j1][i], ROT_A[j] % nbits), f"add_{i}_{j}")
            V1[j] = z1
            if carries:
                weight_bits += carries[:-1]  # exclude overflow carry
            V2[j] = xor2_word(m, V1[j], rotr(R[j2][i], ROT_B[j] % nbits), f"xor2_{i}_{j}")
            V3[j] = V2[j]  # RC dropped (conservative in XOR-diff)

        # t = XOR over lanes of V3[j]
        t = V3[0]
        for j in range(1, lanes):
            t = xor2_word(m, t, V3[j], f"tacc_{i}_{j}")

        if i == 0 and cube_t0_bits > 0:
            for b in range(cube_t0_bits):
                bitval = (cube_t0_value >> b) & 1
                t[b].LB = bitval
                t[b].UB = bitval

        # fo[j] = v3[j] XOR ROTL(t, ROT_G[j])
        FO = [None] * lanes
        for j in range(lanes):
            FO[j] = xor2_word(m, V3[j], rotl(t, ROT_G[j] % nbits), f"fo_{i}_{j}")

        # 3) R_{i+1} = L_i XOR FO
        for j in range(lanes):
            R[j][i+1] = xor2_word(m, L[j][i], FO[j], f"Rnext_{i}_{j}")

    if iterative:
        for j in range(lanes):
            for b in range(nbits):
                m.addConstr(L[j][0][b] == L[j][rounds][b], name=f"iter_L[{j},{b}]")
                m.addConstr(R[j][0][b] == R[j][rounds][b], name=f"iter_R[{j},{b}]")

    if pins:
        for key, val in pins.items():
            mo = re.fullmatch(r'([LR])(\d+)_(\d+)', key)
            if not mo:
                raise ValueError(f"Bad pin key: {key}")
            side, jj, rnd = mo.group(1), int(mo.group(2)), int(mo.group(3))
            word = L[jj][rnd] if side == 'L' else R[jj][rnd]
            for b in range(nbits):
                bit = (val >> b) & 1
                word[b].LB = bit
                word[b].UB = bit

    W = gp.quicksum(weight_bits)

    if mode == "min":
        m.setObjective(W, gp.GRB.MINIMIZE)
    elif mode == "feas_ge":
        m.addConstr(W >= target_weight, name="W_ge")
        m.setObjective(0, gp.GRB.MINIMIZE)
    elif mode == "feas_le":
        m.addConstr(W <= target_weight, name="W_le")
        m.setObjective(0, gp.GRB.MINIMIZE)
    else:
        raise ValueError("mode must be 'min' | 'feas_ge' | 'feas_le'")

    return m, W



#  Restricted rotational XOR-differential check

def add_rot_invariance_word(m: gp.Model, word_bits, k: int, name: str):
    """Enforce word_bits == rotl(word_bits, k)."""
    n = len(word_bits); k %= n
    for b in range(n):
        m.addConstr(word_bits[b] == word_bits[(b - k) % n], name=f"{name}[{b}]")

def build_rotational_xordiff(
    rounds: int,
    k: int,
    nbits: int = 64,
    lanes: int = 8,
    constrain_rounds: Iterable[int] = (0,),
    **kwargs
) -> Tuple[gp.Model, gp.LinExpr]:
 
    m, W = build_fha_xordiff_milp(rounds=rounds, nbits=nbits, lanes=lanes, mode="min", **kwargs)

    for rnd in constrain_rounds:
        for j in range(lanes):
            Lw = [m.getVarByName(f"L{j}_{rnd}[{b}]") for b in range(nbits)]
            Rw = [m.getVarByName(f"R{j}_{rnd}[{b}]") for b in range(nbits)]
            add_rot_invariance_word(m, Lw, k, name=f"rotInv_L{j}_{rnd}")
            add_rot_invariance_word(m, Rw, k, name=f"rotInv_R{j}_{rnd}")

    # Non-trivial diff
    L0sum = gp.quicksum(m.getVarByName(f"L{j}_0[{b}]") for j in range(lanes) for b in range(nbits))
    R0sum = gp.quicksum(m.getVarByName(f"R{j}_0[{b}]") for j in range(lanes) for b in range(nbits))
    m.addConstr(L0sum + R0sum >= 1, name="nontrivial_rot")

    m.setObjective(W, gp.GRB.MINIMIZE)
    return m, W



#  Linear trail MILP (Fu-style addition + correct fan-out)


def xor3_parity0_bit(m: gp.Model, a, b, c, name: str):
    """
    Enforce even parity: a ⊕ b ⊕ c = 0  (equivalently c = a ⊕ b).
    Allowed sums: a+b+c ∈ {0,2}.
    """
    d = m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_d")
    m.addConstr(d >= a, name=f"{name}_dgea")
    m.addConstr(d >= b, name=f"{name}_dgeb")
    m.addConstr(d >= c, name=f"{name}_dgec")
    m.addConstr(a + b + c >= 2*d, name=f"{name}_sumge2d")
    m.addConstr(a + b + c <= 2,   name=f"{name}_sumle2")
    return d

def xor_reduce_word(m: gp.Model, inputs, name: str):
    """out = XOR of all input words (bitwise)."""
    nbits = len(inputs[0])
    out = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_out[{i}]") for i in range(nbits)]
    if len(inputs) == 1:
        for i in range(nbits):
            m.addConstr(out[i] == inputs[0][i], name=f"{name}_copy[{i}]")
        return out

    cur = inputs[0]
    for k in range(1, len(inputs)):
        nxt = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_t{k}[{i}]") for i in range(nbits)]
        for i in range(nbits):
            xor3_parity0_bit(m, cur[i], inputs[k][i], nxt[i], name=f"{name}_xor{k}[{i}]")
        cur = nxt

    for i in range(nbits):
        m.addConstr(out[i] == cur[i], name=f"{name}_outEq[{i}]")
    return out

def fanout_conservation_word(m: gp.Model, node_mask, use_masks, name: str):
    """
    Fan-out constraint for linear trails (mask conservation):
      node_mask = XOR(use_masks[0], use_masks[1], ...).
    """
    xsum = xor_reduce_word(m, use_masks, name=f"{name}_xsum")
    for i in range(len(node_mask)):
        m.addConstr(node_mask[i] == xsum[i], name=f"{name}_cons[{i}]")


def add_linear_addition_fu(
    m: gp.Model,
    gamma,  # output mask Γ
    la,     # input mask Λa
    lb,     # input mask Λb
    name: str,
):
    """
    Fu-style ARX addition linear model: z = x + y mod 2^n.
    Weight = sum_{i=1..n-1} s[i].
    """
    n = len(gamma)
    s = [m.addVar(vtype=gp.GRB.BINARY, name=f"{name}_s[{i}]") for i in range(n + 1)]
    m.addConstr(s[n] == 0, name=f"{name}_sn0")

    for i in range(n):
        m.addConstr(s[i+1] - gamma[i] - la[i] + lb[i] + s[i] >= 0, name=f"{name}_c0[{i}]")
        m.addConstr(s[i+1] + gamma[i] + la[i] - lb[i] - s[i] >= 0, name=f"{name}_c1[{i}]")
        m.addConstr(s[i+1] + gamma[i] - la[i] - lb[i] + s[i] >= 0, name=f"{name}_c2[{i}]")
        m.addConstr(s[i+1] - gamma[i] + la[i] - lb[i] + s[i] >= 0, name=f"{name}_c3[{i}]")
        m.addConstr(s[i+1] + gamma[i] - la[i] + lb[i] - s[i] >= 0, name=f"{name}_c4[{i}]")
        m.addConstr(s[i+1] - gamma[i] + la[i] + lb[i] - s[i] >= 0, name=f"{name}_c5[{i}]")
        m.addConstr(-s[i+1] + gamma[i] + la[i] + lb[i] + s[i] >= 0, name=f"{name}_c6[{i}]")
        m.addConstr(s[i+1] + gamma[i] + la[i] + lb[i] + s[i] <= 4, name=f"{name}_c7[{i}]")

    w = gp.quicksum(s[i] for i in range(1, n))
    return w, s


def build_fha_linear_milp(
    rounds: int = 6,
    nbits: int = 64,
    lanes: int = 8,
    ROT_A: Optional[Iterable[int]] = None,
    ROT_B: Optional[Iterable[int]] = None,
    ROT_G: Optional[Iterable[int]] = None,
    pins: Optional[Dict[str, int]] = None,
    threads: int = 1,
    timelimit: Optional[int] = None,
    env: Optional[gp.Env] = None,
) -> Tuple[gp.Model, gp.LinExpr]:
 

    if ROT_A is None: ROT_A = [63,19,39,15,23,31,11,5]
    if ROT_B is None: ROT_B = [21, 3,31,33,45, 5,17,7]
    if ROT_G is None: ROT_G = [25,45,51, 5,63,47, 7,1]
    ROT_A = list(ROT_A); ROT_B = list(ROT_B); ROT_G = list(ROT_G)
    if len(ROT_A) < lanes: ROT_A = (ROT_A * ((lanes + len(ROT_A)-1)//len(ROT_A)))[:lanes]
    if len(ROT_B) < lanes: ROT_B = (ROT_B * ((lanes + len(ROT_B)-1)//len(ROT_B)))[:lanes]
    if len(ROT_G) < lanes: ROT_G = (ROT_G * ((lanes + len(ROT_G)-1)//len(ROT_G)))[:lanes]

    m = gp.Model("fha_linear", env=env) if env else gp.Model("fha_linear")
    if threads:   m.Params.Threads   = threads
    if timelimit: m.Params.TimeLimit = timelimit

    # State masks
    Lm = [[bits_from_word(m, f"Lm{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]
    Rm = [[bits_from_word(m, f"Rm{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]

    W_terms: List[gp.LinExpr] = []

    for i in range(rounds):
        # --- Fan-out split for each Rm[j][i] (used in multiple places) ---
        Rcopy = [bits_from_word(m, f"Rcopy_{i}_{j}", nbits) for j in range(lanes)]
        Radd  = [bits_from_word(m, f"Radd_{i}_{j}",  nbits) for j in range(lanes)]
        RrotA = [bits_from_word(m, f"RrotA_{i}_{j}", nbits) for j in range(lanes)]
        RrotB = [bits_from_word(m, f"RrotB_{i}_{j}", nbits) for j in range(lanes)]
        for j in range(lanes):
            fanout_conservation_word(m, Rm[j][i], [Rcopy[j], Radd[j], RrotA[j], RrotB[j]], name=f"fan_R_{i}_{j}")

        # 1) L_{i+1} = R_i  (copy use)
        for j in range(lanes):
            eq_word(m, Lm[j][i+1], Rcopy[j], name=f"copyLm_{j}_{i}")

        # 2) Mix-1: v1 = Radd[j] + ROTL(RrotA[j1], ROT_A[j])
        V1m = [bits_from_word(m, f"V1m_{i}_{j}", nbits) for j in range(lanes)]
        for j in range(lanes):
            j1 = (j + 1) % lanes
            la = Radd[j]
            lb = rotl(RrotA[j1], ROT_A[j] % nbits)
            gamma = V1m[j]
            w_add, _ = add_linear_addition_fu(m, gamma=gamma, la=la, lb=lb, name=f"linadd_{i}_{j}")
            W_terms.append(w_add)

        # 3) Mix-2: v2 = v1 XOR ROTR(RrotB[j2], ROT_B[j])
        V2m = [bits_from_word(m, f"V2m_{i}_{j}", nbits) for j in range(lanes)]
        for j in range(lanes):
            j2 = (j + 2) % lanes
            # XOR: for z = x XOR y, mask conservation implies output mask = input masks (transpose relation)
            eq_word(m, V2m[j], V1m[j], name=f"xor_in1_{i}_{j}")
            eq_word(m, V2m[j], rotr(RrotB[j2], ROT_B[j] % nbits), name=f"xor_in2_{i}_{j}")

        # --- Fan-out split for V2m[j] because it is used in (t) and (FO) ---
        V2t  = [bits_from_word(m, f"V2t_{i}_{j}",  nbits) for j in range(lanes)]
        V2fo = [bits_from_word(m, f"V2fo_{i}_{j}", nbits) for j in range(lanes)]
        for j in range(lanes):
            fanout_conservation_word(m, V2m[j], [V2t[j], V2fo[j]], name=f"fan_V2_{i}_{j}")

        # 4) t = XOR over lanes of V2t[j]
        tm = bits_from_word(m, f"tm_{i}", nbits)
        for j in range(lanes):
            eq_word(m, V2t[j], tm, name=f"t_rel_{i}_{j}")

        # --- Fan-out split for t because it is used in every lane's FO ---
        tuse = [bits_from_word(m, f"tuse_{i}_{j}", nbits) for j in range(lanes)]
        fanout_conservation_word(m, tm, tuse, name=f"fan_t_{i}")

        # 5) FO[j] = V2fo[j] XOR ROTL(tuse[j], ROT_G[j])
        FOm = [bits_from_word(m, f"FOm_{i}_{j}", nbits) for j in range(lanes)]
        for j in range(lanes):
            eq_word(m, FOm[j], V2fo[j], name=f"fo_in1_{i}_{j}")
            eq_word(m, FOm[j], rotl(tuse[j], ROT_G[j] % nbits), name=f"fo_in2_{i}_{j}")

        # 6) R_{i+1} = L_i XOR FO
        for j in range(lanes):
            eq_word(m, Rm[j][i+1], Lm[j][i], name=f"Rnext_in1_{i}_{j}")
            eq_word(m, Rm[j][i+1], FOm[j],  name=f"Rnext_in2_{i}_{j}")

    # Pins (optional)
    if pins:
        for key, val in pins.items():
            mo = re.fullmatch(r'([LR])(\d+)_(\d+)', key)
            if not mo:
                raise ValueError(f"Bad pin key: {key}")
            side, jj, rnd = mo.group(1), int(mo.group(2)), int(mo.group(3))
            word = Lm[jj][rnd] if side == 'L' else Rm[jj][rnd]
            for b in range(nbits):
                bit = (val >> b) & 1
                word[b].LB = bit
                word[b].UB = bit

    # Non-trivial input mask (keep ONLY this, to avoid reviewer-attackable restrictions)
    m.addConstr(
        gp.quicksum(Lm[j][0][b] for j in range(lanes) for b in range(nbits)) +
        gp.quicksum(Rm[j][0][b] for j in range(lanes) for b in range(nbits)) >= 1,
        name="nontrivial_input_mask"
    )

    W = gp.quicksum(W_terms)
    m.setObjective(W, gp.GRB.MINIMIZE)
    return m, W


We implemented the MILP model using Gurobi Optimizer v12.0.3 (build v12.0.3rc0, mac64[arm], Darwin 23.0.0 23A344) under an academic license. All experiments were conducted on an Apple M1 system (8 GB RAM) running macOS 14, with the model executed via Jupyter Notebook.

In [ ]:
#for differential cryptoanalysis

def build_FHA_milp(
    rounds: int = 6,
    nbits: int = 64,
    lanes: int = 8,
    ROT_A: Optional[Iterable[int]] = None,
    ROT_B: Optional[Iterable[int]] = None,
    ROT_G: Optional[Iterable[int]] = None,
    iterative: bool = False,
    pins: Optional[Dict[str, int]] = None,
    mode: str = "feas_le",                 
    target_weight: int = 512,              
    cube_t0_bits: int = 0,                
    cube_t0_value: int = 0,                
    threads: int = 8,
    timelimit: Optional[int] = None,
    env: Optional[gp.Env] = None,
) -> Tuple[gp.Model, gp.LinExpr]:
    
   
    if ROT_A is None: ROT_A = [63,19,39,15,23,31,11,5]
    if ROT_B is None: ROT_B = [21, 3,31,33,45, 5,17,7]
    if ROT_G is None: ROT_G = [25,45,51, 5,63,47, 7,1]

    ROT_A = list(ROT_A); ROT_B = list(ROT_B); ROT_G = list(ROT_G)
    if len(ROT_A) < lanes: ROT_A = (ROT_A * ((lanes + len(ROT_A)-1)//len(ROT_A)))[:lanes]
    if len(ROT_B) < lanes: ROT_B = (ROT_B * ((lanes + len(ROT_B)-1)//len(ROT_B)))[:lanes]
    if len(ROT_G) < lanes: ROT_G = (ROT_G * ((lanes + len(ROT_G)-1)//len(ROT_G)))[:lanes]

    m = gp.Model("FHA_xordiff", env=env) if env else gp.Model("FHA_xordiff")
    if threads:   m.Params.Threads   = threads
    if timelimit: m.Params.TimeLimit = timelimit
    L = [[bits_from_word(m, f"L{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]
    R = [[bits_from_word(m, f"R{j}_{i}", nbits) for i in range(rounds+1)] for j in range(lanes)]

    weight_bits = []  

    for i in range(rounds):
       
        for j in range(lanes):
            for b in range(nbits):
                m.addConstr(L[j][i+1][b] == R[j][i][b], name=f"copy_L[{j},{i},{b}]")

       
        V1, V2, V3 = [[] for _ in range(lanes)], [[] for _ in range(lanes)], [[] for _ in range(lanes)]
        for j in range(lanes):
            j1, j2 = (j+1) % lanes, (j+2) % lanes
          
            z1, carries = add_word(m, R[j][i], rotl(R[j1][i], ROT_A[j] % nbits), f"add_{i}_{j}")
            V1[j] = z1
            if carries:
                weight_bits += carries[:-1]  # exclude overflow carry
           
            V2[j] = xor2_word(m, V1[j], rotr(R[j2][i], ROT_B[j] % nbits), f"xor2_{i}_{j}")
            V3[j] = V2[j]  # RC addition is dropped (XOR-diff neutral under this conservative relaxation)

       
        t = V3[0]
        for j in range(1, lanes):
            t = xor2_word(m, t, V3[j], f"tacc_{i}_{j}")

           if i == 0 and cube_t0_bits > 0:
            for b in range(cube_t0_bits):
                bitval = (cube_t0_value >> b) & 1
                t[b].LB = bitval
                t[b].UB = bitval

   
        FO = [None] * lanes
        for j in range(lanes):
            FO[j] = xor2_word(m, V3[j], rotl(t, ROT_G[j] % nbits), f"fo_{i}_{j}")

        for j in range(lanes):
            R[j][i+1] = xor2_word(m, L[j][i], FO[j], f"Rnext_{i}_{j}")

   
    if iterative:
        for j in range(lanes):
            for b in range(nbits):
                m.addConstr(L[j][0][b] == L[j][rounds][b], name=f"iter_L[{j},{b}]")
                m.addConstr(R[j][0][b] == R[j][rounds][b], name=f"iter_R[{j},{b}]")
     if pins:
        for key, val in pins.items():
            mobj = re.fullmatch(r'([LR])(\d+)_(\d+)', key)
            if not mobj:
                raise ValueError(f"Bad pin key: {key}")
            side, jj, rnd = mobj.group(1), int(mobj.group(2)), int(mobj.group(3))
            word = L[jj][rnd] if side == 'L' else R[jj][rnd]
            for b in range(nbits):
                bit = (val >> b) & 1
                word[b].LB = bit
                word[b].UB = bit

   
    W = gp.quicksum(weight_bits)
    if mode == "min":
        m.setObjective(W, gp.GRB.MINIMIZE)
    elif mode == "feas_ge":
        m.addConstr(W >= target_weight, name="W_ge")
        m.setObjective(0, gp.GRB.MINIMIZE)
    elif mode == "feas_le":
        m.addConstr(W <= target_weight, name="W_le")
        m.setObjective(0, gp.GRB.MINIMIZE)
    else:
        raise ValueError("mode must be 'min' | 'feas_ge' | 'feas_le'")

    return m, W
